In [ ]:
# Installing dependencies
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "2019-Oct.csv"

# Loading dataset with pandas kwargs
df = kagglehub.load_dataset(
KaggleDatasetAdapter.PANDAS,
"mkechinov/ecommerce-behavior-data-from-multi-category-store",
file_path,
pandas_kwargs={
"usecols": ["user_id", "event_type", "event_time"],
"nrows": 500_000 # choosing just the enough rows
}
)

/Users/glenhellio/.pyenv/versions/convertiq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/gx/_p6prf1d05bc3xk29_l_p4140000gn/T/ipykernel_95056/3978563033.py:8: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


 28%|██▊       | 456M/1.61G [01:08<05:29, 3.80MB/s] 

In [ ]:
# New columns for date (YYYY-MM-DD) and time (HH:MM:SS)
df['event_time'] = df['event_time'].astype(str)
df['date'] = df['event_time'].str[:10]
df['time'] = df['event_time'].str[11:19]
df.head()

In [ ]:
# Average user session time
# A session = continuous activity per user; gap > 30 min starts a new session

import pandas as pd

df['event_time_dt'] = pd.to_datetime(df['event_time'], utc=True)
df_sorted = df.sort_values(['user_id', 'event_time_dt'])

# Flag new session when gap > 30 min within same user
SESSION_GAP = pd.Timedelta('30min')
df_sorted['prev_time'] = df_sorted.groupby('user_id')['event_time_dt'].shift(1)
df_sorted['new_session'] = (
    (df_sorted['event_time_dt'] - df_sorted['prev_time'] > SESSION_GAP) |
    df_sorted['prev_time'].isna()
)
df_sorted['session_id'] = df_sorted.groupby('user_id')['new_session'].cumsum()

# Session duration = last event - first event per session
session_times = df_sorted.groupby(['user_id', 'session_id'])['event_time_dt'].agg(
    session_start='min', session_end='max'
)
session_times['duration'] = session_times['session_end'] - session_times['session_start']

avg_session = session_times['duration'].mean()
print(f"Average session duration: {avg_session}")

In [ ]:
# Split sessions into quartile buckets (max session = 2h)
bins = [0, 30, 60, 90, 120]
labels = ['0–30 min', '30–60 min', '60–90 min', '90–120 min']

duration_minutes = session_times['duration'].dt.total_seconds() / 60

session_times['quartile'] = pd.cut(duration_minutes, bins=bins, labels=labels, include_lowest=True)

quartile_counts = session_times['quartile'].value_counts().sort_index()
print(quartile_counts)

In [ ]:
# event_type counts per session for Q1 sessions (0–30 min)
q1_sessions = session_times[session_times['quartile'] == '0–30 min'].index

q1_df = df_sorted.set_index(['user_id', 'session_id']).loc[
    df_sorted.set_index(['user_id', 'session_id']).index.isin(q1_sessions)
].reset_index()

q1_df['user_session'] = q1_df['user_id'].astype(str) + '_' + q1_df['session_id'].astype(str)

q1_event_by_session = q1_df.groupby(['user_session', 'event_type']).size().unstack(fill_value=0)
q1_event_by_session